<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Solutions</h2>
<h2>Notebook B04: Probabilistic Forecasting</h2>
</div>

Worked solutions to the 2 exercises in
[Notebook B04: Probabilistic Forecasting](../notebooks/B04_Probabilistic_forecasting.ipynb).

**Try each exercise yourself first.** These notebooks are most useful as a check on your reasoning, and
least useful as something to read straight through. An exercise you attempted and got wrong teaches more
than a solution you agreed with.

Where an exercise asks a question rather than requesting code, the answer is written out under the code
that produces it. Several of them have answers that are more interesting than they look.

The setup cell below reproduces the state the exercises assume, so this notebook runs on its own.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="setup">Setup</h3>
</div>

The rolling-origin forecasts from the notebook. This cell fits the model 24 times and takes about half a
minute.

In [ ]:
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.statespace.sarimax import SARIMAX

sys.path.append("../notebooks")
import nb_config

sns.set_theme(style="whitegrid")

series = pd.read_parquet(nb_config.CDC_TEMP_PATH)["Brandenburg/Berlin"].asfreq("MS")

ORDER, SEASONAL_ORDER, HORIZON = (1, 0, 0), (0, 1, 1, 12), 12


def collect_forecasts(series, n_origins=24, horizon=HORIZON, step=12):
    records = []
    for k in range(n_origins):
        end = len(series) - horizon - (n_origins - 1 - k) * step
        history, actual = series.iloc[:end], series.iloc[end:end + horizon]

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            fitted = SARIMAX(history, order=ORDER, seasonal_order=SEASONAL_ORDER).fit(disp=False)

        forecast = fitted.get_forecast(horizon)
        for h in range(horizon):
            records.append({
                "actual": actual.iloc[h],
                "mean": forecast.predicted_mean.iloc[h],
                "std": forecast.se_mean.iloc[h],
            })
    return pd.DataFrame(records)


evaluations = collect_forecasts(series)
print(f"{len(evaluations)} forecast points from 24 origins")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-1">Exercise 1</h3>
</div>

> Build the same calibration plot for intervals that are too *wide* (multiply `std` by 1.5). Where does the curve sit relative to the diagonal, and why is this failure less dangerous than overconfidence, even though it is equally miscalibrated?

In [ ]:
def empirical_coverage(frame, level):
    """Fraction of outcomes inside the stated interval."""
    z = stats.norm.ppf(0.5 + level / 2)
    return float(((frame["actual"] - frame["mean"]).abs() <= z * frame["std"]).mean())


variants = {
    "Model (x1.0)": evaluations,
    "Overconfident (x0.5)": evaluations.assign(std=evaluations["std"] * 0.5),
    "Underconfident (x1.5)": evaluations.assign(std=evaluations["std"] * 1.5),
}

levels = [0.50, 0.80, 0.95]
coverage_table = pd.DataFrame(
    {name: [empirical_coverage(frame, level) for level in levels]
     for name, frame in variants.items()},
    index=[f"{level:.0%}" for level in levels],
).rename_axis("nominal")

coverage_table.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

grid = np.linspace(0.05, 0.99, 30)
colours = {"Model (x1.0)": "steelblue",
           "Overconfident (x0.5)": "crimson",
           "Underconfident (x1.5)": "seagreen"}

for name, frame in variants.items():
    empirical = [empirical_coverage(frame, level) for level in grid]
    ax.plot(grid, empirical, color=colours[name], linewidth=1.8, label=name)

ax.plot([0, 1], [0, 1], color="black", linestyle="--", linewidth=1.0, label="Perfect")
ax.fill_between([0, 1], [0, 1], [1, 1], color="seagreen", alpha=0.05)
ax.fill_between([0, 1], [0, 0], [0, 1], color="crimson", alpha=0.05)
ax.text(0.55, 0.25, "overconfident\n(dangerous)", fontsize=9, color="crimson")
ax.text(0.15, 0.80, "underconfident\n(wasteful)", fontsize=9, color="seagreen")

ax.set_title("Calibration", fontsize=13, fontweight="bold")
ax.set_xlabel("Nominal coverage")
ax.set_ylabel("Empirical coverage")
ax.legend(loc="lower right", fontsize=9)
ax.grid(linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

**The wide intervals sit above the diagonal**, the mirror image of the overconfident ones below it. At a
nominal 50% they contain the truth 69% of the time; at 95% they contain it **100%** of the time, never
missing once in 288 forecasts.

Both curves are miscalibrated, and by a similar margin. The asymmetry is in what happens next, and it is
entirely about the decisions the interval feeds.

**An overconfident interval understates risk.** A 95% interval that actually holds 68% of the time will be
breached roughly one time in three. Whatever the interval was protecting — a reserve margin, an inventory
buffer, a capital requirement — is sized for an event that occurs far more often than budgeted. The
failures are **correlated with the moments that matter**, because intervals are breached when the world
does something unusual, which is exactly when the buffer is needed.

**An underconfident interval overstates risk.** The reserve is larger than necessary, the buffer is
expensive, the forecast is less useful than it could be. That is a real cost, and it is a cost you can see
in the budget rather than one that arrives as a surprise.

The distinction is between **errors you pay for continuously and know about**, and **errors you do not pay
for until they arrive all at once**. A 100% coverage at the 95% level is visibly too conservative and
someone will eventually tighten it. A 68% coverage at the 95% level looks fine until the year it does not.

None of which makes the wide intervals correct. They are wrong, they should be fixed, and the fix is the
same either way: measure coverage and scale accordingly. But if you must be wrong in one direction while
you work it out, be wrong in the direction whose cost you can see.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-2">Exercise 2</h3>
</div>

> The overconfident forecast claims half the true uncertainty. Find the multiplier that minimises CRPS by trying a range of values between 0.5 and 2.0. Does the best score land at 1.0, and what does it mean if it does not?

In [ ]:
def crps_normal(actual, mean, std):
    """CRPS for a normal predictive distribution, in closed form."""
    actual, mean, std = map(np.asarray, (actual, mean, std))
    z = (actual - mean) / std
    return float(np.mean(
        std * (z * (2 * stats.norm.cdf(z) - 1) + 2 * stats.norm.pdf(z) - 1 / np.sqrt(np.pi))
    ))


# Rounded, because np.arange produces values like 1.0000000000000002 that will
# not match a .loc lookup of 1.0
multipliers = np.round(np.arange(0.5, 2.01, 0.05), 2)
scores = pd.Series(
    {
        multiplier: crps_normal(
            evaluations["actual"], evaluations["mean"], evaluations["std"] * multiplier
        )
        for multiplier in multipliers
    }
)

best_multiplier = scores.idxmin()

print(f"Best multiplier: {best_multiplier:.2f}   (CRPS {scores.min():.4f})")
print(f"CRPS at 1.00:    {scores.loc[1.0]:.4f}")
print()
print(scores.loc[[0.5, 0.8, 0.9, 1.0, 1.1, 1.25, 1.5, 2.0]].round(4).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))

ax.plot(scores.index, scores.values, color="steelblue", linewidth=1.8)
ax.axvline(1.0, color="black", linestyle="--", linewidth=1.0, label="true scale")
ax.scatter([best_multiplier], [scores.min()], color="crimson", s=70, zorder=3,
           label=f"minimum at {best_multiplier:.2f}")

ax.set_title("CRPS against the scale of the predictive distribution",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Multiplier applied to the standard deviation")
ax.set_ylabel("CRPS")
ax.legend()
ax.grid(linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

**Yes, the minimum lands at exactly 1.00.** The curve is smooth, with a clear single minimum at the
model's own uncertainty estimate, rising in both directions.

That is the defining property of a **proper** scoring rule, and this exercise is a demonstration of it. A
proper score is minimised, in expectation, by stating your true beliefs. You cannot improve it by claiming
more confidence than you have, and you cannot improve it by hedging: shrinking the intervals to 0.5 costs
you (1.236 against 1.147), and inflating them to 2.0 costs you more (1.315).

Compare that with the metrics in the first half of the notebook. MAE could not distinguish the honest
forecast from the overconfident one at all, because their point forecasts were identical. CRPS grades the
whole distribution and is optimised only by getting the whole distribution right.

**What would it mean if the minimum were not at 1.0?** It would be a direct, quantitative statement that
the model has the wrong uncertainty, and it would say by how much:

- A minimum **below 1.0** means the model is underconfident: its intervals are wider than its errors
  warrant, and multiplying the standard errors by that factor would improve every probabilistic forecast
  it makes.
- A minimum **above 1.0** means it is overconfident, which is the more common and the more dangerous case.

That makes the exercise a usable calibration tool rather than a demonstration. Fit a model, collect
forecasts across many origins, scan the multiplier, and if the minimum is not near 1.0, scale the
intervals by it. It is the probabilistic equivalent of noticing a bias in the point forecast and
subtracting it — and here, reassuringly, there is nothing to correct.

---

Back to [Notebook B04](../notebooks/B04_Probabilistic_forecasting.ipynb), or on to
[Notebook C01](../notebooks/C01_Feature_engineering.ipynb).